RAG

In [27]:
#imports
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq
from typing import List,Annotated,TypedDict
from langgraph.graph import StateGraph,END
from langchain_core.messages import HumanMessage,SystemMessage,BaseMessage
import os

In [28]:
#preparing a text splitters

char_splitter = RecursiveCharacterTextSplitter(chunk_size=128,chunk_overlap=0)

In [29]:
directory = "./Documents"
fileNames = os.listdir(directory)
chunks = []
for file in fileNames:
    with open(f"{directory}/{file}") as f:
        filetext = f.read()
        texts = char_splitter.split_text(filetext)
        for chunkindex,text in enumerate(texts):
            chunks.append(Document(page_content=text,metadata={
                "chunkindex":chunkindex,
                "sourcefile":file
            }))

In [30]:
for chunk in chunks[:2]:
    print("------------")
    print("Page Content")
    print(chunk.page_content)
    print("Metadata")
    print(chunk.metadata)

------------
Page Content
# Development Team Internal Directory

## Team Scope

Handles:
Metadata
{'chunkindex': 0, 'sourcefile': 'Development Team.txt'}
------------
Page Content
* Feature development (frontend & backend)
* Bug fixing and issue resolution
* API development and integration
Metadata
{'chunkindex': 1, 'sourcefile': 'Development Team.txt'}


In [31]:
# if embeddings:
#     del embeddings
embeddings = HuggingFaceEmbeddings()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1648.77it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [32]:
vector_store = FAISS.from_documents(documents=chunks,embedding=embeddings)

In [33]:
#ReRanker
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

# Your base retriever (FAISS)
base_retriever = vector_store.as_retriever(search_kwargs={"k": 5})

# Initialize BGE reranker
model = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-v2-m3")
compressor = CrossEncoderReranker(model=model, top_n=3)

# Wrap retriever
reranker = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=base_retriever
)

# Retrieve
# docs = reranker.invoke("How much years of experience Ankit Sharma have?")   

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 1157.70it/s]


In [34]:
def get_surrounding_chunks(retrieved_chunk, all_chunks, window=5):
    idx = retrieved_chunk.metadata["chunkindex"]
    start = max(0, idx - window)
    end = min(len(all_chunks), idx + window + 1)
    return all_chunks[start:end]

In [35]:
from dotenv import load_dotenv
load_dotenv()
GROQ_LLM_API_KEY = os.getenv("GROQ_LLM_API_KEY")
GEMINI_LLM_API_KEY= os.getenv("GEMINI_LLM_API_KEY")


In [36]:
# llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash",temperature=0.7,api_key=GEMINI_LLM_API_KEY)
llm = ChatGroq(model="llama-3.3-70b-versatile",temperature=0.7,api_key=GROQ_LLM_API_KEY)

In [37]:
class GraphState(TypedDict):
    messages:Annotated[List[BaseMessage],lambda x,y:x+y]

In [46]:
def fetchRelatedChunks(prompt):
    chunks_from_reranker = reranker.invoke(prompt)
    context_chunks = []
    for retrievedchunk in chunks_from_reranker:
        context_chunks = context_chunks + get_surrounding_chunks(retrievedchunk,chunks)
    return context_chunks
    
def prepareContextFromChunks(chunks=[]):
    return "".join([f"Source:{content.metadata["sourcefile"]}, content : {content.page_content}" for content in chunks])

In [71]:
def call_reranker(state:GraphState):
    messages = state.get("messages")
    last_message = messages[-1]
    prompt = last_message.content
    chunks_from_reranker = fetchRelatedChunks(prompt)
    context_message = prepareContextFromChunks(chunks_from_reranker)
    context_section = f"""
RELEVANT INFORMATION RETRIEVED:
--------------------------------
{context_message}
--------------------------------
Use this information naturally in conversation if relevant.
""" if context_message else "No additional context retrieved."
    system_message = SystemMessage(content=context_message)
    messages = messages + [system_message]
    return {"messages":messages}
    

In [87]:
def call_llm(state:GraphState):
    messages = state.get("messages")
    system_msg = SystemMessage(content=f"""
    You are a friendly and helpful assistant.
    BEHAVIOR:
    - Chat naturally and conversationally with the user.
    - If retrieved context is available and relevant, weave it into your response naturally.
    - Ask follow-up questions to better understand the user's need.
    - Do NOT say "based on the context" or "according to retrieved data" — just talk naturally.
    - If context is partially relevant, use what applies and ask the user to clarify the rest.
    - If context is not relevant, just chat normally — do NOT force it.
    - If context doesnot seem relevant , ask further questions
    - Use tools only when you need live/real-time data (orders, items, etc).
    - Keep the response short and detail oriented 
    - Donot overask to the user
    - Donot overexplain some information
    - If multiple domains are involved, mention about them in short
    - Donot assume and pass on same information
    - If user keep on asking same question , try asking further points to understand the concern
    - Donot add information about the context , names forcefully 
    - without any confirmation donot involve any person or suggest any person
    - only recomment someone when you will be completely sure
    """)
    if len(messages) < 4:
        messages = messages + [system_msg]
    
    response = llm.invoke(messages)
    return {"messages":[response]}

In [88]:
workflow = StateGraph(GraphState)
workflow.add_node("call_reranker",call_reranker)
workflow.add_node("call_llm",call_llm)
workflow.set_entry_point("call_reranker")
workflow.set_finish_point("call_llm")
workflow.add_edge("call_reranker","call_llm")
app = workflow.compile()

In [89]:
inputs = {
    "messages":[]
}
def process(prompt):
    print(f"Human : {prompt}")
    query = HumanMessage(prompt)
    messages = inputs.get("messages")
    messages = messages + [query]
    inputs["messages"] = messages
    response = app.invoke(inputs)
    lastmessage = response.get("messages")[-1]
    print(f"AI : {lastmessage.content}\n\n")


In [90]:
human_prompts = [
    "A customer has reported an issue with our product.",

    "The issue is related to the dashboard module.",

    "The UI is not loading properly and some components are missing.",

    "I can see some warnings in the browser console related to rendering.",

    "This issue started after the last deployment.",

    "We recently added new dashboard widgets in that release.",

    "The issue is happening only in the production environment.",

    "It is affecting multiple users across the system.",

    "Backend APIs seem to be working fine and data is coming correctly.",

    "Yes, please involve the relevant team members to resolve this issue."
]
for question in human_prompts:
    process(question)
    

Human : A customer has reported an issue with our product.
AI : What kind of issue is the customer experiencing? Is it related to the UI, backend, or something else? That way, I can try to point you in the direction of the right team or person to help resolve it.


Human : The issue is related to the dashboard module.
AI : Based on the Development Team directory, I would recommend assigning this issue to either Rohan Mehta (Senior Frontend Developer) or Sneha Kapoor (Full Stack Developer), as they both have expertise in frontend development and may be able to assist with dashboard-related issues. 

Rohan Mehta has 7 years of experience and is an expert in React, UI performance, and state management, which could be relevant to dashboard issues. 
Sneha Kapoor has 6 years of experience and is a full-stack developer with expertise in frontend + backend integration, which could also be helpful in resolving dashboard issues.

It would be best to reach out to one of them to report the issue a

In [91]:
if embeddings:
    del embeddings
if model:
    del model
# import torch
# print(torch.cuda.is_available()) s